# 11. Tuning to the junction counts: lanes + pedestrians

**Question:** which lever brings the model's junction volumes to the JICA 2011-12 counts?
**Target:** GEH < 5 at all 6 tuning junctions (Chabahil, Gausala, Kalimati, New Baneshwor, Old Baneshwor, Thapathali), on the mean volume over 5 seeds, with road speeds still near the JICA speed survey (Ring Road 24.0, inner main roads 15.3 km/h).

The other 3 counted junctions (Balaju, Kalanki, Koteshwor) were set aside by a fixed random draw before any tuning run and are not shown here; they are checked only once a configuration has been chosen.

**This notebook:** Round 5's best setting still left Kalimati too high. A check of its approach roads found that they have no lane count in OpenStreetMap, so the importer gave them its generic default: 3-4 lanes per direction for a trunk road, where the tagged trunk roads of Kathmandu have a median of 2 (secondary: default 2, tagged median 1). Here untagged main roads get the median of the tagged roads of their class (trunk 2, primary 2, secondary 1). Also new: pedestrian friction on side streets and non-trunk main roads (trunk roads such as the Ring Road have footpaths and are exempt). Links within 50 m of an OpenStreetMap pedestrian street (the pedestrianised core: Basantapur, Thamel, Ason) get 15 km/h; the top 10% of the remaining links by OpenStreetMap shops, amenities and offices within 100 m along the whole link, per km, get 30 km/h. Both rules apply to the whole city, come from OpenStreetMap only, and were fixed before any run; they replace round 5's midpoint rule. Every configuration runs on seeds 1-5, next to the unchanged baseline. Volumes are measured in the second hour of a two-hour steady peak.

In [ ]:
import glob, os, shutil, subprocess, sys, zipfile

def find_project(marker):
    """Copy the project tree that contains `marker` from /kaggle/input to /kaggle/working/sim."""
    hits = sorted(glob.glob(f"/kaggle/input/**/{marker}", recursive=True),
                  key=lambda h: ("build-demand" not in h, len(h)))   # prefer step 1's copy
    if not hits:                                        # dataset delivered as a zip
        for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
            zipfile.ZipFile(z).extractall("/kaggle/working/_unzipped")
        hits = sorted(glob.glob(f"/kaggle/working/_unzipped/**/{marker}", recursive=True), key=len)
    src = hits[0][: -len(marker)].rstrip("/")
    shutil.copytree(src, "/kaggle/working/sim", dirs_exist_ok=True)
    os.chdir("/kaggle/working/sim")
    print("project from", src)

def run(*cmd, tries=1):
    """Run a script, streaming its output; retry resumable drivers if a worker is killed."""
    for attempt in range(tries):
        p = subprocess.Popen([sys.executable, "-u", *cmd], stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end="")
        if p.wait() == 0:
            return
        print(f"exit code {p.returncode}, attempt {attempt + 1} of {tries}")
    raise RuntimeError(" ".join(cmd))

find_project("scripts/tune_counts.py")

In [ ]:
!pip install -q uxsim==1.14.2

In [ ]:
run("scripts/build_minor_ways.py")
run("scripts/build_od_jica.py", "--data", "data/jica_net")

In [ ]:
run("scripts/tune_counts.py", "--idea", "net6", "--seeds", "5", "--jobs", "3", tries=3)

In [ ]:
run("scripts/tune_counts.py", "--idea", "net6", "--summary-only", "--radius", "100")